In [1]:
import glob
import os
import random
import time
import numpy as np
import time
from string import Template

import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.ppo import CnnPolicy, MlpPolicy
# from stable_baselines3.common.evaluation import evaluate_policy
# from stable_baselines3.dqn import CnnPolicy, MlpPolicy
from sb3_contrib import RecurrentPPO

from pettingzoo.mpe import simple_tag_v3

from ray import train, tune
from ray.train import Checkpoint
from ray.tune.schedulers import PopulationBasedTraining

In [2]:
class MetricLogger(BaseCallback):
    def __init__(self, log_frequency=100, verbose=0):
        super(MetricLogger, self).__init__(verbose=verbose)
        self.verbose=verbose
        self.log_frequency = log_frequency
        self.train_losses=[]
        self.train_value_losses=[]

    def _on_step(self) -> bool:
        if self.n_calls % self.log_frequency == 0:
            if (self.verbose == 1):
                # print(f"iterations: {self.model.logger.name_to_value['train/n_updates']}")
                # print(f"ep_rew_mean: {self.model.logger.name_to_value['train/ep_rew_mean']}")
                # print(f"policy_loss: {self.model.logger.name_to_value['train/policy_loss']}")
                # print(f"value_loss: {self.model.logger.name_to_value['train/value_loss']}")
                # print(f"entropy_loss: {self.model.logger.name_to_value['train/entropy_loss']}")
                # print("--------------------------------")
                train.report({
                    "loss" : self.model.logger.name_to_value['train/loss'],
                    "value_loss" : self.model.logger.name_to_value['train/value_loss'],
                })

        return True

In [3]:
models = {
    "PPO" : PPO,
    "RecPPO" : RecurrentPPO,
    "DQN" : DQN,
}

filename = Template("${env}_model-${model}_policy-${policy}_tsteps-${timesteps}_${timestamp}")

In [4]:
def train_model(env_fn, seed: int | None = 0, config = None, tb_logs_path="../tb_logs", **env_kwargs):
    # Train a single model to play as each agent in an AEC environment
    env = env_fn.parallel_env(**env_kwargs)

    env.reset(seed=seed)

    print(f"Starting training on {str(env.metadata['name'])}, {config['model_name']}, {config['model_policy']}, {config['steps']}.")
    env = ss.multiagent_wrappers.pad_observations_v0(env)
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, 8, num_cpus=1, base_class="stable_baselines3")

    # Model
    model = models[config['model_name']](
        config['model_policy'],
        env,
        verbose=1,
        tensorboard_log=tb_logs_path,
        **config['model_kwargs']
    )
    #model = PPO("MlpPolicy", env, verbose=1, batch_size=256,)
    #model = RecurrentPPO("MlpLstmPolicy", env, verbose=1, batch_size=256,)
    #model = DQN("MlpPolicy", env, verbose=1, batch_size=256,)

    # Train
    path = filename.substitute(
        env=env.unwrapped.metadata.get('name'),
        model=config['model_name'],
        policy=config['model_policy'],
        timesteps=config['steps'],
        timestamp=time.strftime('%Y%m%d-%H%M%S'),
    )
    metric_logger = MetricLogger(verbose=1)
    model.learn(total_timesteps=config['steps'], tb_log_name=path, callback=metric_logger)

    model.save(path)
    # model.save(f"{env.unwrapped.metadata.get('name')}_model-{model_name}_policy-{model_policy}_tsteps-{steps}_{time.strftime('%Y%m%d-%H%M%S')}")

    print("Model has been saved.")

    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")

    env.close()

In [5]:
# Set render_mode to None to reduce training time
batch_size = 2048
env_fn = simple_tag_v3
runs = [
    {
        "steps" : 1000000,
        "model_name" : "PPO",
        "model_policy" : "MlpPolicy",
        "batch_size" : 2048,
        "model_kwargs" : dict(
            n_steps=np.random.randint(32, 5000), # Horizon range (32,5000), PPO recommended n_steps*n_envs
            batch_size=np.random.randint(1, 128)*32, #
            n_epochs=np.random.randint(3, 30),
            clip_range=tune.choice([0.1, 0.2, 0.3]),
            target_kl=tune.uniform(0.003, 0.03), # loguniform?
            gae_lambda=tune.uniform(0.9, 1), # .95 default
            vf_coef=tune.loguniform(0.5, 1),
            ent_coef=tune.loguniform(1e-10, 0.01),
            learning_rate=tune.loguniform(5e-6, 0.003)
        )
    },
    # {
    #     "steps" : 1000000,
    #     "model_name" : "RecPPO",
    #     "model_policy" : "MlpLstmPolicy",
    #     "batch_size" : 2048,
    # },
    # {
    #     "steps" : 1000000,
    #     "model_name" : "DQN",
    #     "model_policy" : "MlpPolicy",
    #     "batch_size" : 512,
    # },
]

In [6]:
results = []
for run in runs:
    train_model(
        env_fn,
        seed=0,
        config=run,
        **env_kwargs
    )
    scheduler = PopulationBasedTraining(
        time_attr="training_iteration",
        perturbation_interval=5,
        metric="loss",
        mode="max",
    )
    tuner = tune.Tuner(
        trainable=train_model,
        stop={"training_iteration": 50},
        tune_config=tune.TuneConfig(
            scheduler=scheduler,
            num_samples=4
        )
    )
    results.append(tuner.fit())

NameError: name 'env_kwargs' is not defined

In [ ]:

def eval_(env_fn, num_games: int = 10000, render_mode: str | None = None, model_name="PPO", policy_path = None, **env_kwargs):
    # Evaluate a trained agent vs a random agent
    env = env_fn.env(render_mode=render_mode, **env_kwargs)
    env.metadata['render_fps'] = 60
    print(
        f"\nStarting evaluation on {str(env.metadata['name'])} (num_games={num_games}, render_mode={render_mode})"
    )

    try:
        if policy_path is None:
            policy = max(
                glob.glob(f"{env.metadata['name']}_model-{model_name}*.zip"), key=os.path.getctime
            )
        else:
            policy = policy_path

    except ValueError:
        print("Policy not found.")
        exit(0)

    model = models[model_name].load(policy)
    #model = PPO.load(latest_policy)
    #model = RecurrentPPO.load(latest_policy)
    #model = DQN.load(latest_policy)

    rewards = {agent: 0 for agent in env.possible_agents}

    # Note: we evaluate here using an AEC environments, to allow for easy A/B testing against random policies
    # For example, we can see here that using a random agent for archer_0 results in less points than the trained agent
    for i in range(num_games):
        env.reset(seed=i)
        env.action_space(env.possible_agents[0]).seed(i)
        
        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()
            if agent == 'agent_0':
                obs=np.append(obs, [0,0])
            #print(obs)
            if render_mode== 'human':
                time.sleep(0.01)
            for agent in env.agents:
                rewards[agent] += env.rewards[agent]

            if termination or truncation:
                break
            else:
                if agent == env.possible_agents[0]:
                    act = env.action_space(agent).sample()
                else:
                    act = model.predict(obs, deterministic=True)[0]
            env.step(act)
    env.close()

    avg_reward = sum(rewards.values()) / len(rewards.values())
    avg_reward_per_agent = {
        agent: rewards[agent] / num_games for agent in env.possible_agents
    }
    print(f"Avg reward: {avg_reward}")
    print("Avg reward per agent, per game: ", avg_reward_per_agent)
    print("Full rewards: ", rewards)
    return avg_reward


In [ ]:
# eval_(env_fn, num_games=10, render_mode=None, model_name=model_name, policy_path=None, **env_kwargs)
# eval_(env_fn, num_games=10, render_mode=None, model_name=model_name, policy_path=None, **env_kwargs)
# eval_(env_fn, num_games=10, render_mode=None, model_name=model_name, policy_path=None, **env_kwargs)
# eval_(env_fn, num_games=10, render_mode=None, model_name=model_name, policy_path=None, **env_kwargs)

import glob
import os
import time
import numpy as np
import time

import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3.ppo import CnnPolicy, MlpPolicy
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.evaluation import evaluate_policy

from pettingzoo.mpe import simple_tag_v3
env_fn = simple_tag_v3
print(env_fn)

In [ ]:
env_kwargs = dict(num_good=1, num_adversaries=3, num_obstacles=2, max_cycles=100, continuous_actions=False )
eval_(env_fn, num_games=3,  render_mode='human', model_name="PPO", **env_kwargs)

In [ ]:
eval_(env_fn, num_games=3,  render_mode='human', model_name="RecPPO", **env_kwargs)

In [ ]:
eval_(env_fn, num_games=3,  render_mode='human', model_name="DQN", **env_kwargs)